In [1]:
import cv2 as cv
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from contour_detection import find_hand_contour
from feature_extraction import extract_features, create_hand_mask
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import joblib

In [2]:
RPS_DATA = Path("rps2")
X = []
y= []
for class_folder in RPS_DATA.iterdir():
    #If if class_folder is actually a folder
    if not class_folder.is_dir():
        continue
    #Get label/gesture of the images of that folder
    label = class_folder.name
    for img_path in class_folder.iterdir():
        #Read the image
        img = cv.imread(str(img_path))
        #Entire image is our region of interest
        mask = create_hand_mask(img)
        hand_contour = find_hand_contour(mask)
        features = extract_features(hand_contour,img)
        if features is None:
            continue
        X.append(features)
        y.append(label)
X= pd.DataFrame(X)
y = np.array(y)

In [3]:
X.head()

,defect_count,solidity,aspect_ratio,extent,area_ratio,hull_area_ratio
0,3.0,0.642973,0.673333,0.503927,0.339311,0.527722
1,1.0,0.911268,0.683761,0.679995,0.282878,0.310422
2,2.0,0.842590,0.593750,0.637811,0.275761,0.327278
3,4.0,0.763877,0.737069,0.534533,0.235622,0.308456
4,4.0,0.712198,0.894118,0.491710,0.317644,0.446006


In [4]:
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)
print(label_encoder.classes_)

['paper' 'rock' 'scissors']


In [5]:
X_train, X_test, y_train, y_test = train_test_split(X,y_encoded,test_size=0.2,random_state=42,stratify=y_encoded)

In [10]:
model = XGBClassifier(n_estimators=500,        # more trees = model can learn more patterns
    max_depth=4,             # slightly deeper trees than 3
    learning_rate=0.03,      # lower learning rate = slower but often better learning
    subsample=0.85,          # use 85% of rows per tree to reduce overfitting
    colsample_bytree=0.85,   # use 85% of features per tree to reduce overfitting

    min_child_weight=2,      # prevents weak/unreliable splits
    gamma=0.1,               # only split if it gives useful improvement
    reg_alpha=0.05,          # L1 regularization
    reg_lambda=2.0,          # L2 regularization

    eval_metric="mlogloss",
    random_state=42
)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test,y_pred,target_names=label_encoder.classes_))

Accuracy: 0.8133333333333334
              precision    recall  f1-score   support

       paper       0.80      0.79      0.80       180
        rock       0.79      0.80      0.79       166
    scissors       0.85      0.85      0.85       179

    accuracy                           0.81       525
   macro avg       0.81      0.81      0.81       525
weighted avg       0.81      0.81      0.81       525



In [8]:
joblib.dump(model,"xgboost_rps.pkl")
joblib.dump(label_encoder,"label_encoder.pkl")

['label_encoder.pkl']